In [1]:
from transformers import pipeline
import pandas as pd

review = pd.read_excel("product_reviews.xlsx")

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sample_reviews = review["ReviewText"].head(3).tolist()

for text in sample_reviews:
    result = classifier(text)[0]
    print(f"Review: {text}")
    print(f"Prediction: {result['label']} | Score: {result['score']:.3f}\n")

C:\Users\corne\Job\packt_python_4e\Analyze Image Data\analyze_image_data\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 1378.11it/s]


Review: Very disappointed, it stopped working after a week.
Prediction: NEGATIVE | Score: 1.000

Review: Very useful and well-designed, worth every penny.
Prediction: POSITIVE | Score: 1.000

Review: Poor quality and not as described.
Prediction: NEGATIVE | Score: 1.000



In [2]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]

    # Step 1: similarity between queries and keys
    scores = np.matmul(Q, K.T) / np.sqrt(d_k)

    # Step 2: normalize into attention weights
    weights = softmax(scores)

    # Step 3: weighted combination of values
    output = np.matmul(weights, V)

    return output, weights

# Example with 3 tokens and 4-dimensional vectors
Q = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0]
])

K = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0]
])

V = np.array([
    [10.0, 0.0],
    [0.0, 10.0],
    [5.0, 5.0]
])

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention weights:")
print(np.round(weights, 3))

print("\nAttention output:")
print(np.round(output, 3))

Attention weights:
[[0.506 0.186 0.307]
 [0.186 0.506 0.307]
 [0.274 0.274 0.452]]

Attention output:
[[6.601 3.399]
 [3.399 6.601]
 [5.    5.   ]]


In [3]:
def split_heads(X, num_heads):
    seq_len, d_model = X.shape
    d_head = d_model // num_heads
    return X.reshape(seq_len, num_heads, d_head)

def combine_heads(X):
    seq_len, num_heads, d_head = X.shape
    return X.reshape(seq_len, num_heads * d_head)

# Example input with 3 tokens and d_model = 4
X = np.array([
    [1.0, 0.5, 0.2, 0.1],
    [0.3, 1.0, 0.4, 0.2],
    [0.8, 0.7, 0.1, 0.9]
])

num_heads = 2
heads = split_heads(X, num_heads)

head_outputs = []

for h in range(num_heads):
    Q_h = heads[:, h, :]
    K_h = heads[:, h, :]
    V_h = heads[:, h, :]
    out_h, _ = scaled_dot_product_attention(Q_h, K_h, V_h)
    head_outputs.append(out_h)

stacked = np.stack(head_outputs, axis=1)
final_output = combine_heads(stacked)

print("Multi-head output:")
print(np.round(final_output, 3))

Multi-head output:
[[0.738 0.707 0.233 0.405]
 [0.676 0.751 0.233 0.409]
 [0.719 0.721 0.214 0.483]]


In [4]:
# from transformers import AutoTokenizer, AutoModelForCausalLM

# model_name = "your-chosen-open-llm"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)

# prompt = "Explain the difference between traditional NLP and LLMs."
# inputs = tokenizer(prompt, return_tensors="pt")

# output_ids = model.generate(
#     **inputs,
#     max_new_tokens=80
# )

# generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
# print(generated_text)

In [5]:
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# 1) Prepare the data
data = review[["ReviewText", "SentimentLabel"]].dropna().copy()

label_map = {"negative": 0, "neutral": 1, "positive": 2}
data["label"] = (
    data["SentimentLabel"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(label_map)
)

data = data.dropna(subset=["label"])[["ReviewText", "label"]]
data["label"] = data["label"].astype(int)

# 2) Split the data
train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False)
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True), preserve_index=False)

# 3) Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(batch["ReviewText"], truncation=True)

train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["ReviewText"])
test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["ReviewText"])

# 4) Load model
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="transfer_learning_output",
    eval_strategy="epoch",
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none"
)

# 5) Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 1202.55it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\corne\Job\packt_python_4e\Analyze Image Data\analyze_image_data\Lib\site-packages\torch\utils\data\datalo

Epoch,Training Loss,Validation Loss
1,No log,0.882747


TrainOutput(global_step=10, training_loss=1.0197441101074218, metrics={'train_runtime': 5.1024, 'train_samples_per_second': 15.679, 'train_steps_per_second': 1.96, 'total_flos': 275288722128.0, 'train_loss': 1.0197441101074218, 'epoch': 1.0})

In [6]:
sample_texts = [
    "The product quality is excellent and delivery was fast.",
    "The item works, but the packaging was disappointing.",
    "This is the worst purchase I have made this year."
]

encoded = tokenizer(
    sample_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

model.eval()
with torch.no_grad():
    outputs = model(**encoded)

predictions = outputs.logits.argmax(dim=1).tolist()

for text, pred in zip(sample_texts, predictions):
    print(f"Review: {text}")
    print(f"Predicted sentiment: {id2label[pred]}\n")

Review: The product quality is excellent and delivery was fast.
Predicted sentiment: positive

Review: The item works, but the packaging was disappointing.
Predicted sentiment: negative

Review: This is the worst purchase I have made this year.
Predicted sentiment: negative



In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def run_prompt(prompt, max_new_tokens=20):
    inputs = tokenizer(prompt.strip(), return_tensors="pt", truncation=True)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def normalize_label(text):
    text = text.strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if text.startswith(label) or label in text:
            return label
    return "unknown"

review_text = "The delivery was fast and the product quality is excellent."

prompt = f"""
Classify the sentiment of the following review as positive, negative, or neutral.
Return only one label.

Review: {review_text}
Sentiment:
"""

prediction = normalize_label(run_prompt(prompt, max_new_tokens=10))
print(prediction)    

Loading weights: 100%|██████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 963.65it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


positive


In [8]:
few_shot_prompt = """
Classify each review as positive, negative, or neutral.
Return only one label.

Review: The phone is amazing and the battery lasts all day.
Sentiment: positive

Review: The item arrived damaged and stopped working.
Sentiment: negative

Review: The product is acceptable, but nothing special.
Sentiment: neutral

Review: The packaging was neat and the product works very well.
Sentiment:
"""

prediction = normalize_label(run_prompt(few_shot_prompt, max_new_tokens=10))
print(prediction)

positive


In [9]:
review_text = "The sound quality is great, but the battery life is too short."

prompt = f"""
Analyze the following product review.

Return the result in this format:
sentiment: <positive/negative/neutral>
reason: <short explanation>

Review: {review_text}
"""

print(run_prompt(prompt, max_new_tokens=50))

negative


In [10]:
def build_sentiment_prompt(review_text):
    return f"""
Classify the sentiment of the following review as positive, negative, or neutral.
Return only one label.

Review: The delivery was fast and the product quality is excellent.
Sentiment: positive

Review: The item arrived damaged and stopped working.
Sentiment: negative

Review: The product is acceptable, but nothing special.
Sentiment: neutral

Review: {review_text}
Sentiment:
""".strip()

sample_reviews = review["ReviewText"].dropna().head(3)

for text in sample_reviews:
    prediction = normalize_label(run_prompt(build_sentiment_prompt(text), max_new_tokens=10))
    print(f"Review: {text}")
    print(f"Prediction: {prediction}\n")

Review: Very disappointed, it stopped working after a week.
Prediction: negative

Review: Very useful and well-designed, worth every penny.
Prediction: positive

Review: Poor quality and not as described.
Prediction: negative



In [11]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, TaskType, get_peft_model

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1
    }

training_args = TrainingArguments(
    output_dir="lora_output",
    save_strategy="no",
    eval_strategy="epoch",      # evaluate during training
    logging_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

results = trainer.evaluate()
print(results)

model.save_pretrained("distilbert_sentiment_lora")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 2019.68it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,1.095511,1.062003,0.500000,0.250000,0.500000,0.333333


C:\Users\corne\Job\packt_python_4e\Analyze Image Data\analyze_image_data\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.0620027780532837, 'eval_accuracy': 0.5, 'eval_precision_weighted': 0.25, 'eval_recall_weighted': 0.5, 'eval_f1_weighted': 0.3333333333333333, 'eval_runtime': 0.1846, 'eval_samples_per_second': 108.315, 'eval_steps_per_second': 16.247, 'epoch': 1.0}


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def run_prompt(prompt, max_new_tokens=20):
    inputs = tokenizer(prompt.strip(), return_tensors="pt", truncation=True)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

Loading weights: 100%|█████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 3553.19it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [13]:
meeting_notes = """
The analytics team reviewed Q1 performance. Revenue grew by 8 percent compared with the previous quarter.
Customer retention improved in the enterprise segment, but churn increased for smaller accounts.
The team agreed to investigate onboarding friction and prepare a dashboard for weekly retention monitoring.
"""

prompt = f"""
Summarize the following meeting notes in 2 short sentences.

Notes:
{meeting_notes}

Summary:
"""

print(run_prompt(prompt, max_new_tokens=80))

A company's analytics team reviewed Q1 earnings.


In [ ]:
policy_text = """
Employees may carry over up to 5 unused vacation days into the next calendar year.
Any additional unused days will expire at the end of December unless approved by management.
"""

question = "How many unused vacation days can employees carry over?"

prompt = f"""
Answer the question using only the context below.
If the answer is not stated in the context, say "Not found in the context."

Context:
{policy_text}

Question:
{question}

Answer:
"""

print(run_prompt(prompt, max_new_tokens=60))

In [14]:
prompt = f"""
Summarize the following meeting notes for an executive audience.
Focus only on business impact and next actions.
Return 3 bullet points.

Notes:
{meeting_notes}

Summary:
"""

print(run_prompt(prompt, max_new_tokens=100))

The company's analytics team reviewed Q1 earnings and revenue.


In [15]:
feedback = "The app is useful, but the login process is very confusing and slow."

prompt = f"""
Classify the following customer feedback into one of these categories:
bug, usability, billing, feature_request, general_feedback

Return only one label.

Feedback: {feedback}
Label:
"""

print(run_prompt(prompt, max_new_tokens=10))

usability


In [16]:
email_text = """
Hello team, I was charged twice for my March subscription.
My account email is user123@example.com.
Please refund the extra payment as soon as possible.
"""

prompt = f"""
Extract the following fields from the email.

Return exactly in this format:
issue_type: <text>
customer_email: <text>
urgency: <low/medium/high>

Email:
{email_text}
"""

print(run_prompt(prompt, max_new_tokens=80))

issue_type: text> customer_email: text> customer_email: text> customer_email: text> customer_email: text> customer_email: text> customer_email: text> customer_email: text> customer_email: text>


In [18]:
from sentence_transformers import SentenceTransformer, util
import torch

embedder = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "LoRA reduces fine-tuning cost by training a small number of additional parameters.",
    "Prompt engineering improves model behavior by changing the input rather than the weights.",
    "Question answering systems return answers from a context passage.",
    "Summarization creates shorter versions of long documents."
]

query = "How can I lower the cost of adapting a large language model?"

doc_embeddings = embedder.encode(documents, convert_to_tensor=True)
query_embedding = embedder.encode(query, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, doc_embeddings)[0]
top_k = torch.topk(scores, k=2)

for score, idx in zip(top_k.values, top_k.indices):
    print(documents[idx])
    print("Score:", float(score), "\n")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1640.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LoRA reduces fine-tuning cost by training a small number of additional parameters.
Score: 0.34179389476776123 

Summarization creates shorter versions of long documents.
Score: 0.32797181606292725 



In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from sentence_transformers import SentenceTransformer, util

qa_model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def answer_from_context(question, context):
    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True
    )

    qa_model.eval()
    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_idx = torch.argmax(outputs.start_logits, dim=1).item()
    end_idx = torch.argmax(outputs.end_logits, dim=1).item()

    if end_idx < start_idx:
        end_idx = start_idx

    answer_ids = inputs["input_ids"][0][start_idx:end_idx + 1]
    return qa_tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

documents = [
    "ARIF supports dashboards, questionnaires, descriptive analytics, and report generation.",
    "LoRA is a parameter-efficient fine-tuning method that freezes the base model and trains small adapters.",
    "Semantic search uses vector similarity to retrieve relevant text from a document collection."
]

query = "What is LoRA used for?"

doc_embeddings = embedder.encode(documents, convert_to_tensor=True)
query_embedding = embedder.encode(query, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, doc_embeddings)[0]
best_idx = torch.argmax(scores).item()
retrieved_context = documents[best_idx]

answer = answer_from_context(query, retrieved_context)

print("Retrieved context:", retrieved_context)
print("Answer:", answer)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2853.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved context: LoRA is a parameter-efficient fine-tuning method that freezes the base model and trains small adapters.
Answer: a parameter - efficient fine - tuning method that freezes the base model and trains small adapters
